## Creating 'Gold' database for reporting purposes.

In [0]:
CREATE DATABASE IF NOT EXISTS gold;

## Conversion Funnel 

In [0]:
CREATE OR REPLACE TABLE gold.fact_conversion_funnel
USING DELTA
AS

WITH session_level_made_it AS (
  SELECT
    website_session_id,
    MAX(CASE WHEN category_url = 'homepage' THEN 1 ELSE 0 END) AS saw_homepage,
    MAX(CASE WHEN category_url = 'product_list' THEN 1 ELSE 0 END) AS saw_product_list,
    MAX(CASE WHEN category_url = 'product_page' THEN 1 ELSE 0 END) AS saw_product_page,
    MAX(CASE WHEN category_url = 'cart' THEN 1 ELSE 0 END) AS saw_cart,
    MAX(CASE WHEN category_url = 'checkout' THEN 1 ELSE 0 END) AS saw_checkout,
    MAX(CASE WHEN category_url = 'order_confirmation' THEN 1 ELSE 0 END) AS saw_thankyou,
    MAX(CASE WHEN category_url = 'promo_pages' THEN 1 ELSE 0 END) AS saw_promo
  FROM silver.website_pageviews
  GROUP BY 1
)
SELECT * FROM session_level_made_it;

In [0]:
SELECT
    COUNT(website_session_id) AS total_sessions,
    SUM(saw_homepage) AS to_home,
    SUM(saw_product_list) AS to_products,
    SUM(saw_cart) AS to_cart,
    SUM(saw_thankyou) AS to_orders,
    ROUND(SUM(saw_thankyou) / COUNT(website_session_id) * 100, 2) AS overall_conversion_rate
FROM gold.fact_conversion_funnel;

### Marketing Channel Performance

In [0]:
CREATE OR REPLACE TABLE gold.mkt_channel_performance
USING DELTA
AS

SELECT 
  COUNT(*) AS total_sessions,
  ws.utm_source,
  ws.utm_campaign,
  
  COUNT(o.order_id) AS total_orders,
  SUM(o.items_purchased) AS total_items_purchased,
  SUM(o.cogs_usd) AS total_cogs_usd,
  SUM(o.price_usd) AS total_revenue,
  SUM(o.gross_margin) AS total_margin,
  ROUND(SUM(o.revenue_tax), 2) AS total_revenue_tax,

  ROUND(COUNT(o.order_id) / NULLIF(COUNT(*), 0) * 100, 2) AS conversion_rate,
  ROUND(SUM(o.gross_margin) / NULLIF(COUNT(o.order_id), 0)) AS average_order_value_per_order


FROM silver.website_sessions ws
LEFT JOIN silver.orders o
ON ws.website_session_id = o.website_session_id
GROUP BY ws.utm_source, ws.utm_campaign;

In [0]:
SELECT 
  *
FROM gold.mkt_channel_performance;

### Monthly Sales Growth

In [0]:
CREATE OR REPLACE TABLE gold.monthly_sales_growth
USING DELTA
AS

WITH monthly_revenue AS (

  SELECT
    COUNT(order_id) AS total_orders,
    DATE_TRUNC('month', created_at) AS trx_date,
    SUM(price_usd) AS curr_month_revenue
  FROM silver.orders
  GROUP BY trx_date
),

prev_month_revenue AS (
  SELECT
    *,
    LAG(curr_month_revenue) OVER (ORDER BY trx_date) AS prev_month_revenue
  FROM monthly_revenue
)
SELECT
  *,
  ROUND((curr_month_revenue - prev_month_revenue) / NULLIF(prev_month_revenue, 0) * 100, 2) AS growth_rate
FROM prev_month_revenue;


  

In [0]:
SELECT * FROM gold.monthly_sales_growth;

### Cross-Selling Analysis

In [0]:
CREATE OR REPLACE TABLE gold.product_cross_sell
USING DELTA 
AS

WITH cross_sell_summary AS (  
  SELECT
    o.primary_product_id AS product_a_id,
    oi.product_id AS product_b_id,
    COUNT(*) AS cross_sell_count
  FROM silver.orders o   
  JOIN silver.order_items oi
    ON o.order_id = oi.order_id 
  WHERE oi.is_primary_item = 0 
  GROUP BY o.primary_product_id, oi.product_id
)
SELECT 
    pa.product_name AS primary_product,
    pb.product_name AS cross_sold_product,
    css.cross_sell_count
FROM cross_sell_summary css
JOIN silver.products pa ON css.product_a_id = pa.product_id  
JOIN silver.products pb ON css.product_b_id = pb.product_id  
ORDER BY cross_sell_count DESC;

In [0]:
SELECT * FROM gold.product_cross_sell